# Triple Barrier Labelling

https://paperswithbacktest.com/course/triple-barrier-method

**Generate initial Triple Barrier labels**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Define project directories
PROJECT_DIR = Path(r"C:\UTAR - Master\Master Project\Project\Main Project")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

# Load feature-engineered data
df = pd.read_csv(
    PROCESSED_DIR / "lseg_feature_engineered.csv",
    parse_dates=["Date"]
)

df = df.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

# Define initial Triple Barrier parameters
VOLATILITY_WINDOW = 5
HOLDING_PERIOD = 5
UPPER_MULTIPLIER = 2.0
LOWER_MULTIPLIER = 2.0

volatility_column = f"volatility_{VOLATILITY_WINDOW}"

# Validate input data
assert len(df) == 37798
assert df["Ticker"].nunique() == 13
assert not df.duplicated(["Ticker", "Date"]).any()
assert volatility_column in df.columns

# Generate labels within uninterrupted historical segments
label_records = []

for (ticker, segment_id), stock in df.groupby(
    ["Ticker", "Segment_ID"],
    sort=False
):

    stock = stock.sort_values("Date").reset_index(drop=True)

    close = stock["Close"].to_numpy(dtype=float)
    volatility = stock[volatility_column].to_numpy(dtype=float)
    dates = stock["Date"].to_numpy()

    for i in range(len(stock) - HOLDING_PERIOD):

        sigma = volatility[i]

        if not np.isfinite(sigma) or sigma <= 0:
            continue

        entry_price = close[i]

        upper_barrier = entry_price * (
            1 + UPPER_MULTIPLIER * sigma
        )

        lower_barrier = entry_price * (
            1 - LOWER_MULTIPLIER * sigma
        )

        # Examine future closing prices only
        future_close = close[
            i + 1:i + HOLDING_PERIOD + 1
        ]

        future_dates = dates[
            i + 1:i + HOLDING_PERIOD + 1
        ]

        label = 0
        event_position = HOLDING_PERIOD - 1
        barrier_type = "Time"

        # Identify the first barrier reached
        for j, future_price in enumerate(future_close):

            if future_price >= upper_barrier:
                label = 1
                event_position = j
                barrier_type = "Upper"
                break

            if future_price <= lower_barrier:
                label = -1
                event_position = j
                barrier_type = "Lower"
                break

        exit_price = future_close[event_position]

        label_records.append({
            "Ticker": ticker,
            "Segment_ID": segment_id,
            "Date": dates[i],
            "Event_End_Date": future_dates[event_position],
            "Entry_Price": entry_price,
            "Exit_Price": exit_price,
            "Volatility": sigma,
            "Upper_Barrier": upper_barrier,
            "Lower_Barrier": lower_barrier,
            "Holding_Days": event_position + 1,
            "Barrier_Type": barrier_type,
            "Label": label,
            "Event_Return": exit_price / entry_price - 1
        })

# Assemble labelled observations
tb_labels = pd.DataFrame(label_records)

# Add stock metadata
stock_metadata = df[
    ["Ticker", "Name", "Country"]
].drop_duplicates("Ticker")

tb_labels = tb_labels.merge(
    stock_metadata,
    on="Ticker",
    how="left",
    validate="many_to_one"
)

# Validate generated labels
assert tb_labels["Label"].isin([-1, 0, 1]).all()
assert tb_labels["Event_End_Date"].gt(tb_labels["Date"]).all()
assert tb_labels["Holding_Days"].between(
    1, HOLDING_PERIOD
).all()

assert not tb_labels.duplicated(
    ["Ticker", "Date"]
).any()

# Summarise label distribution
label_names = {
    -1: "Down",
    0: "Flat",
    1: "Up"
}

label_summary = (
    tb_labels.groupby(["Country", "Label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[-1, 0, 1], fill_value=0)
    .rename(columns=label_names)
)

label_percent = (
    label_summary.div(
        label_summary.sum(axis=1),
        axis=0
    ) * 100
).round(2)

print("=== TRIPLE BARRIER CONFIGURATION ===")
print("Volatility lookback:", VOLATILITY_WINDOW)
print("Holding period:", HOLDING_PERIOD)
print("Upper multiplier:", UPPER_MULTIPLIER)
print("Lower multiplier:", LOWER_MULTIPLIER)

print("\n=== LABEL COUNTS BY MARKET ===")
display(label_summary)

print("\n=== LABEL PERCENTAGES BY MARKET ===")
display(label_percent)

print("\n=== TOTAL LABELLED OBSERVATIONS ===")
print(len(tb_labels))

print("\n=== BARRIER TYPE COUNTS ===")
display(tb_labels["Barrier_Type"].value_counts())

print("\n=== SAMPLE LABELLED OBSERVATIONS ===")
display(
    tb_labels[
        [
            "Ticker",
            "Date",
            "Event_End_Date",
            "Entry_Price",
            "Upper_Barrier",
            "Lower_Barrier",
            "Barrier_Type",
            "Label",
            "Holding_Days"
        ]
    ].head(15)
)

# Save exploratory labels
output_path = (
    PROCESSED_DIR / "triple_barrier_initial_5d_5h_2x.csv"
)

tb_labels.to_csv(output_path, index=False)

print("\nInitial labels saved:", output_path)
print("Completed.")

=== TRIPLE BARRIER CONFIGURATION ===
Volatility lookback: 5
Holding period: 5
Upper multiplier: 2.0
Lower multiplier: 2.0

=== LABEL COUNTS BY MARKET ===


Label,Down,Flat,Up
Country,,,
Malaysia,7523,13568,7501
Singapore,2334,3772,2768



=== LABEL PERCENTAGES BY MARKET ===


Label,Down,Flat,Up
Country,,,
Malaysia,26.31,47.45,26.23
Singapore,26.30,42.51,31.19



=== TOTAL LABELLED OBSERVATIONS ===
37466

=== BARRIER TYPE COUNTS ===


Barrier_Type
Time     17340
Upper    10269
Lower     9857
Name: count, dtype: int64


=== SAMPLE LABELLED OBSERVATIONS ===


,Ticker,Date,Event_End_Date,Entry_Price,Upper_Barrier,Lower_Barrier,Barrier_Type,Label,Holding_Days
0,1015.KL,2015-01-09,2015-01-16,6.42,6.614312,6.225688,Time,0,5
1,1015.KL,2015-01-12,2015-01-19,6.44,6.617180,6.262820,Time,0,5
2,1015.KL,2015-01-13,2015-01-20,6.45,6.535253,6.364747,Time,0,5
3,1015.KL,2015-01-14,2015-01-21,6.41,6.515378,6.304622,Time,0,5
4,1015.KL,2015-01-15,2015-01-20,6.40,6.445752,6.354248,Upper,1,3
5,1015.KL,2015-01-16,2015-01-20,6.41,6.457530,6.362470,Upper,1,2
6,1015.KL,2015-01-19,2015-01-21,6.44,6.492969,6.387031,Upper,1,2
7,1015.KL,2015-01-20,2015-01-22,6.48,6.544610,6.415390,Upper,1,2
8,1015.KL,2015-01-21,2015-01-22,6.51,6.550444,6.469556,Upper,1,1
9,1015.KL,2015-01-22,2015-01-29,6.59,6.642234,6.537766,Lower,-1,5



Initial labels saved: C:\UTAR - Master\Master Project\Main Project\data\processed\triple_barrier_initial_5d_5h_2x.csv
Completed.


**Compare Triple Barrier configurations**

In [2]:
configurations = [
    (5, 5, 2.0),
    (5, 10, 2.0),
    (10, 5, 2.0),
    (10, 10, 2.0)
]

def generate_triple_barrier_labels(data, vol_window, horizon, multiplier):
    records = []
    volatility_column = f"volatility_{vol_window}"

    for (ticker, segment_id), stock in data.groupby(
        ["Ticker", "Segment_ID"],
        sort=False
    ):
        stock = stock.sort_values("Date").reset_index(drop=True)

        close = stock["Close"].to_numpy(dtype=float)
        volatility = stock[volatility_column].to_numpy(dtype=float)
        dates = stock["Date"].to_numpy()

        for i in range(len(stock) - horizon):
            sigma = volatility[i]

            if not np.isfinite(sigma) or sigma <= 0:
                continue

            entry_price = close[i]
            upper_barrier = entry_price * (1 + multiplier * sigma)
            lower_barrier = entry_price * (1 - multiplier * sigma)

            future_prices = close[i + 1:i + horizon + 1]

            upper_hits = np.flatnonzero(
                future_prices >= upper_barrier
            )

            lower_hits = np.flatnonzero(
                future_prices <= lower_barrier
            )

            first_upper = (
                int(upper_hits[0]) if len(upper_hits) else horizon
            )

            first_lower = (
                int(lower_hits[0]) if len(lower_hits) else horizon
            )

            if first_upper < first_lower:
                label = 1
                event_position = first_upper
                barrier_type = "Upper"

            elif first_lower < first_upper:
                label = -1
                event_position = first_lower
                barrier_type = "Lower"

            else:
                label = 0
                event_position = horizon - 1
                barrier_type = "Time"

            records.append({
                "Ticker": ticker,
                "Country": stock.loc[i, "Country"],
                "Date": dates[i],
                "Event_End_Date": dates[i + event_position + 1],
                "Label": label,
                "Barrier_Type": barrier_type,
                "Holding_Days": event_position + 1,
                "Volatility_Window": vol_window,
                "Holding_Period": horizon,
                "Multiplier": multiplier
            })

    return pd.DataFrame(records)


# Generate labels for each configuration
comparison_results = []

for vol_window, horizon, multiplier in configurations:
    labels = generate_triple_barrier_labels(
        df,
        vol_window=vol_window,
        horizon=horizon,
        multiplier=multiplier
    )

    labels["Configuration"] = (
        f"V{vol_window}_H{horizon}_M{multiplier:g}"
    )

    comparison_results.append(labels)

tb_comparison = pd.concat(
    comparison_results,
    ignore_index=True
)

# Verify the original configuration was reproduced
baseline = tb_comparison.loc[
    tb_comparison["Configuration"] == "V5_H5_M2"
]

assert len(baseline) == len(tb_labels)

assert (
    baseline["Label"].value_counts().sort_index()
    .equals(
        tb_labels["Label"].value_counts().sort_index()
    )
)

# Summarise labels by configuration and market
summary = (
    tb_comparison
    .groupby(["Configuration", "Country", "Label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[-1, 0, 1], fill_value=0)
    .rename(columns={-1: "Down", 0: "Flat", 1: "Up"})
    .reset_index()
)

summary["Total"] = (
    summary["Down"] + summary["Flat"] + summary["Up"]
)

for label in ["Down", "Flat", "Up"]:
    summary[f"{label}_Pct"] = (
        summary[label] / summary["Total"] * 100
    ).round(2)

# Summarise event holding periods
holding_summary = (
    tb_comparison
    .groupby(["Configuration", "Country"])
    .agg(
        Mean_Holding_Days=("Holding_Days", "mean"),
        Median_Holding_Days=("Holding_Days", "median")
    )
    .reset_index()
)

summary = summary.merge(
    holding_summary,
    on=["Configuration", "Country"],
    how="left"
)

print("=== TRIPLE BARRIER CONFIGURATION COMPARISON ===")

display(
    summary[
        [
            "Configuration",
            "Country",
            "Total",
            "Down_Pct",
            "Flat_Pct",
            "Up_Pct",
            "Mean_Holding_Days"
        ]
    ].round(2)
)

print("\n=== LABEL COUNTS ===")

display(
    summary[
        [
            "Configuration",
            "Country",
            "Down",
            "Flat",
            "Up",
            "Total"
        ]
    ]
)

# Save configuration comparison
summary.to_csv(
    RESULTS_DIR / "triple_barrier_configuration_comparison.csv",
    index=False
)

tb_comparison.to_csv(
    PROCESSED_DIR / "triple_barrier_configuration_labels.csv",
    index=False
)

print("\nCompleted.")

=== TRIPLE BARRIER CONFIGURATION COMPARISON ===


,Configuration,Country,Total,Down_Pct,Flat_Pct,Up_Pct,Mean_Holding_Days
0,V10_H10_M2,Malaysia,28337,35.65,28.49,35.86,6.06
1,V10_H10_M2,Singapore,8844,34.06,22.37,43.58,5.69
2,V10_H5_M2,Malaysia,28474,23.54,52.55,23.91,3.99
3,V10_H5_M2,Singapore,8859,23.39,47.30,29.31,3.91
4,V5_H10_M2,Malaysia,28452,36.97,26.26,36.76,5.66
5,V5_H10_M2,Singapore,8859,35.51,20.08,44.41,5.29
6,V5_H5_M2,Malaysia,28592,26.31,47.45,26.23,3.78
7,V5_H5_M2,Singapore,8874,26.30,42.51,31.19,3.70



=== LABEL COUNTS ===


,Configuration,Country,Down,Flat,Up,Total
0,V10_H10_M2,Malaysia,10103,8073,10161,28337
1,V10_H10_M2,Singapore,3012,1978,3854,8844
2,V10_H5_M2,Malaysia,6703,14964,6807,28474
3,V10_H5_M2,Singapore,2072,4190,2597,8859
4,V5_H10_M2,Malaysia,10520,7472,10460,28452
5,V5_H10_M2,Singapore,3146,1779,3934,8859
6,V5_H5_M2,Malaysia,7523,13568,7501,28592
7,V5_H5_M2,Singapore,2334,3772,2768,8874



Completed.


**Compare Triple Barrier labels across development periods**

In [3]:
# Define chronological development periods
periods = {
    "Training_2015_2022": (
        pd.Timestamp("2015-01-01"),
        pd.Timestamp("2022-12-31")
    ),
    "Validation_2023_2024": (
        pd.Timestamp("2023-01-01"),
        pd.Timestamp("2024-12-31")
    )
}

period_records = []

# Summarise labels within each development period
for period_name, (start_date, end_date) in periods.items():

    period_data = tb_comparison.loc[
        (tb_comparison["Date"] >= start_date)
        & (tb_comparison["Date"] <= end_date)
        & (tb_comparison["Event_End_Date"] <= end_date)
    ].copy()

    counts = (
        period_data
        .groupby(["Configuration", "Country", "Label"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=[-1, 0, 1], fill_value=0)
    )

    for (configuration, country), row in counts.iterrows():

        total = int(row.sum())

        period_records.append({
            "Period": period_name,
            "Configuration": configuration,
            "Country": country,
            "Total": total,
            "Down": int(row[-1]),
            "Flat": int(row[0]),
            "Up": int(row[1]),
            "Down_Pct": round(row[-1] / total * 100, 2),
            "Flat_Pct": round(row[0] / total * 100, 2),
            "Up_Pct": round(row[1] / total * 100, 2)
        })

temporal_summary = pd.DataFrame(period_records)

# Verify class counts
assert (
    temporal_summary[["Down", "Flat", "Up"]]
    .sum(axis=1)
    .eq(temporal_summary["Total"])
    .all()
)

# Compare label distributions between periods
temporal_comparison = temporal_summary.pivot(
    index=["Configuration", "Country"],
    columns="Period",
    values=["Down_Pct", "Flat_Pct", "Up_Pct"]
)

print("=== LABEL DISTRIBUTION BY DEVELOPMENT PERIOD ===")

display(
    temporal_summary[
        [
            "Period",
            "Configuration",
            "Country",
            "Total",
            "Down_Pct",
            "Flat_Pct",
            "Up_Pct"
        ]
    ].sort_values(
        ["Country", "Configuration", "Period"]
    )
)

print("\n=== TEMPORAL COMPARISON ===")

display(temporal_comparison)

# Save temporal analysis
temporal_summary.to_csv(
    RESULTS_DIR / "triple_barrier_temporal_distribution.csv",
    index=False
)

print("\nCompleted.")

=== LABEL DISTRIBUTION BY DEVELOPMENT PERIOD ===


,Period,Configuration,Country,Total,Down_Pct,Flat_Pct,Up_Pct
0,Training_2015_2022,V10_H10_M2,Malaysia,19020,36.28,28.92,34.81
8,Validation_2023_2024,V10_H10_M2,Malaysia,4834,32.19,28.20,39.62
2,Training_2015_2022,V10_H5_M2,Malaysia,19124,24.17,52.27,23.56
10,Validation_2023_2024,V10_H5_M2,Malaysia,4852,21.64,53.01,25.35
4,Training_2015_2022,V5_H10_M2,Malaysia,19145,38.04,26.24,35.72
12,Validation_2023_2024,V5_H10_M2,Malaysia,4830,32.88,26.15,40.97
6,Training_2015_2022,V5_H5_M2,Malaysia,19247,27.11,46.85,26.04
14,Validation_2023_2024,V5_H5_M2,Malaysia,4849,23.94,48.05,28.01
1,Training_2015_2022,V10_H10_M2,Singapore,5964,36.17,22.62,41.21
9,Validation_2023_2024,V10_H10_M2,Singapore,1481,33.22,22.15,44.63



=== TEMPORAL COMPARISON ===


Down_Pct                       \
Period                  Training_2015_2022 Validation_2023_2024   
Configuration Country                                             
V10_H10_M2    Malaysia               36.28                32.19   
              Singapore              36.17                33.22   
V10_H5_M2     Malaysia               24.17                21.64   
              Singapore              24.80                22.58   
V5_H10_M2     Malaysia               38.04                32.88   
              Singapore              37.67                35.56   
V5_H5_M2      Malaysia               27.11                23.94   
              Singapore              27.92                25.47   

                                  Flat_Pct                       \
Period                  Training_2015_2022 Validation_2023_2024   
Configuration Country                                             
V10_H10_M2    Malaysia               28.92                28.20   
              Singapore              22.62                22.15   
V10_H5_M2     Malaysia               52.27                53.01   
              Singapore              47.26                48.12   
V5_H10_M2     Malaysia               26.24                26.15   
              Singapore              20.45                19.20   
V5_H5_M2      Malaysia               46.85                48.05   
              Singapore              42.19                44.22   

                                    Up_Pct                       
Period                  Training_2015_2022 Validation_2023_2024  
Configuration Country                                            
V10_H10_M2    Malaysia               34.81                39.62  
              Singapore              41.21                44.63  
V10_H5_M2     Malaysia               23.56                25.35  
              Singapore              27.93                29.30  
V5_H10_M2     Malaysia               35.72                40.97  
              Singapore              41.88                45.23  
V5_H5_M2      Malaysia               26.04                28.01  
              Singapore              29.89                30.31


Completed.


**Prepare chronological model development datasets**

In [4]:
# Define candidate model features
feature_columns = [
    "returns",
    "returns_lag_1",
    "returns_lag_2",
    "momentum_5",
    "momentum_10",
    "daily_range",
    "candle_body",
    "sma_5_ratio",
    "sma_10_ratio",
    "sma_20_ratio",
    "volatility_5",
    "volatility_10",
    "volatility_20",
    "rsi",
    "macd",
    "macd_signal",
    "macd_diff",
    "volume_change",
    "volume_ratio_20",
    "bb_high",
    "bb_low",
    "bb_width",
    "skewness_20"
]

# Select historical features and segment identifiers
feature_data = df[
    ["Ticker", "Date", "Segment_ID"] + feature_columns
].copy()

# Join labels with features
model_data = tb_comparison.merge(
    feature_data,
    on=["Ticker", "Date"],
    how="inner",
    validate="many_to_one"
)

# Define chronological period boundaries
TRAIN_END = pd.Timestamp("2022-12-31")
VALIDATION_START = pd.Timestamp("2023-01-01")
VALIDATION_END = pd.Timestamp("2024-12-31")

# Identify training events with mature labels
train_mask = (
    (model_data["Date"] <= TRAIN_END)
    & (model_data["Event_End_Date"] <= TRAIN_END)
)

# Identify validation events with mature labels
validation_mask = (
    (model_data["Date"] >= VALIDATION_START)
    & (model_data["Date"] <= VALIDATION_END)
    & (model_data["Event_End_Date"] <= VALIDATION_END)
)

# Retain observations with complete model features
complete_features = (
    model_data[feature_columns].notna().all(axis=1)
)

development_data = model_data.loc[
    (train_mask | validation_mask)
    & complete_features
].copy()

# Assign training and validation periods
development_data["Split"] = np.where(
    development_data["Date"] <= TRAIN_END,
    "Train_2015_2022",
    "Validation_2023_2024"
)

# Verify chronological separation
train_data = development_data.loc[
    development_data["Split"] == "Train_2015_2022"
]

validation_data = development_data.loc[
    development_data["Split"] == "Validation_2023_2024"
]

assert (train_data["Event_End_Date"] <= TRAIN_END).all()

assert (
    validation_data["Date"] >= VALIDATION_START
).all()

assert (
    validation_data["Event_End_Date"] <= VALIDATION_END
).all()

assert (
    development_data["Date"] < pd.Timestamp("2025-01-01")
).all()

assert development_data[feature_columns].notna().all().all()

assert not development_data.duplicated(
    ["Configuration", "Ticker", "Date"]
).any()

# Summarise development observations
development_summary = (
    development_data
    .groupby(["Configuration", "Country", "Split"])
    .agg(
        Observations=("Label", "size"),
        Down=("Label", lambda x: (x == -1).sum()),
        Flat=("Label", lambda x: (x == 0).sum()),
        Up=("Label", lambda x: (x == 1).sum()),
        First_Date=("Date", "min"),
        Last_Date=("Date", "max")
    )
    .reset_index()
)

print("=== MODEL DEVELOPMENT DATA SUMMARY ===")

display(development_summary)

print("\n=== OBSERVATIONS BY CONFIGURATION ===")

display(
    development_data.groupby(
        ["Configuration", "Split"]
    ).size().unstack(fill_value=0)
)

print("\n=== DEVELOPMENT DATA VALIDATION ===")

print("Candidate features:", len(feature_columns))
print("Configurations:", development_data["Configuration"].nunique())
print("Total development records:", len(development_data))
print("Training records:", len(train_data))
print("Validation records:", len(validation_data))
print("Records from 2025 onward:", int(
    (development_data["Date"] >= "2025-01-01").sum()
))

# Save model development dataset
development_data.to_csv(
    PROCESSED_DIR / "triple_barrier_model_development.csv",
    index=False
)

development_summary.to_csv(
    RESULTS_DIR / "triple_barrier_development_summary.csv",
    index=False
)

print("\nCompleted.")

=== MODEL DEVELOPMENT DATA SUMMARY ===


,Configuration,Country,Split,Observations,Down,Flat,Up,First_Date,Last_Date
0,V10_H10_M2,Malaysia,Train_2015_2022,18434,6752,5264,6418,2015-02-24,2022-12-28
1,V10_H10_M2,Malaysia,Validation_2023_2024,4811,1551,1352,1908,2023-01-03,2024-12-27
2,V10_H10_M2,Singapore,Train_2015_2022,5895,2128,1324,2443,2015-02-18,2022-12-28
3,V10_H10_M2,Singapore,Validation_2023_2024,1481,492,328,661,2023-01-03,2024-12-20
4,V10_H5_M2,Malaysia,Train_2015_2022,18531,4530,9634,4367,2015-02-24,2022-12-28
5,V10_H5_M2,Malaysia,Validation_2023_2024,4829,1048,2555,1226,2023-01-03,2024-12-27
6,V10_H5_M2,Singapore,Train_2015_2022,5906,1464,2782,1660,2015-02-18,2022-12-28
7,V10_H5_M2,Singapore,Validation_2023_2024,1488,336,716,436,2023-01-03,2024-12-23
8,V5_H10_M2,Malaysia,Train_2015_2022,18427,7066,4790,6571,2015-02-24,2022-12-29
9,V5_H10_M2,Malaysia,Validation_2023_2024,4802,1580,1251,1971,2023-01-03,2024-12-30



=== OBSERVATIONS BY CONFIGURATION ===


Split,Train_2015_2022,Validation_2023_2024
Configuration,,
V10_H10_M2,24329,6292
V10_H5_M2,24437,6317
V5_H10_M2,24324,6281
V5_H5_M2,24427,6309



=== DEVELOPMENT DATA VALIDATION ===
Candidate features: 23
Configurations: 4
Total development records: 122716
Training records: 97517
Validation records: 25199
Records from 2025 onward: 0

Completed.
